# Step 4 — Player Registry & Golden Set

Two **annotation** tasks. Unlike the pipeline notebooks, these need your eyes, and each produces a small hand-checked artifact that everything downstream reads.

### Part A — Handedness registry → `derived/meta/players.json`

The `side` column says which *end of the table* a player stands at. It says nothing about which hand holds the bat, and the two are independent.

This matters more than it sounds. A left-hander at the right end produces a mirror-image skeleton to a right-hander at the same end. Without correction the model must learn both patterns separately from limited data — and per the source paper, **all the left-handers are in the test videos**, so they land entirely in folds F and G. Uncorrected, those two folds evaluate a body configuration the model has never seen in training.

### Part B — Golden set → `derived/meta/golden_set.json`

~40 hand-verified strokes. When Stage 4 returns 45% instead of 65%, this tells you in ten minutes whether it's a label bug or a model problem.

---

**GPU runtime required** (the detector runs on every sampled frame).

**Time:** ~15 min compute, ~25 min of your attention.


## 1 · Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2 · Config & load Stage 0 output

In [4]:
BASE = "/content/drive/MyDrive/tt_coach"

# --- sampling knobs ----------------------------------------------------------
N_HAND_FRAMES  = 3     # serve frames per player per video (handedness sheet)
N_GOLDEN       = 40    # strokes to hand-verify
GOLDEN_STRIP   = 5     # frames per golden stroke (spread across the window)
CROP_PAD       = 0.35  # expand player bbox by this fraction (arm + bat reach)

import json, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

BASE   = Path(BASE)
META   = BASE / "derived/meta"
VIDEOS = BASE / "raw/videos"
SHEETS = META / "review_sheets"
SHEETS.mkdir(parents=True, exist_ok=True)

def load(stem):
    p = META / f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META / f"{stem}.csv")

strokes = load("strokes")
folds   = json.loads((META / "folds.json").read_text())
VIDEO_IDS = sorted(strokes.video_id.unique(),
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))

random.seed(42); np.random.seed(42)

print(f"{len(strokes)} strokes across {len(VIDEO_IDS)} videos")
print(f"player-instances to review: {len(VIDEO_IDS) * 2}")

1457 strokes across 12 videos
player-instances to review: 24


## 3 · Load detector

In [5]:
!pip install -q rtmlib onnxruntime-gpu ultralytics lightgbm shap pyarrow 2>&1 | tail -3
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."

det = YOLO(str(BASE / "models/detector/best.pt"))
det.to("cuda")
print(f"detector classes: {det.names}")

PLAYER_CLS = [k for k, v in det.names.items() if v.lower() == "player"][0]
TABLE_CLS  = [k for k, v in det.names.items() if v.lower() == "table"][0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 42.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
detector classes: {0: 'player', 1: 'table'}


## 4 · Frame access & player resolution

**Which detection is the "left" player?** The detector finds people but doesn't know who's who, and the umpire is usually also in frame. Heuristic: take the detected table's horizontal centre, then pick the player box furthest left of it and the one furthest right. The umpire sits near the centre and behind the table, so the extremes are reliably the two athletes.

Every crop is drawn with its resolved side label so you can spot a mis-assignment immediately.

In [6]:
class VideoReader:
    """Sequential-ish frame access. Seeking a 120fps h264 file over the Drive
    mount is slow, so always request frames in ascending order per video."""

    def __init__(self, path):
        self.cap = cv2.VideoCapture(str(path))
        self.n = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def at(self, idx):
        idx = int(max(0, min(idx, self.n - 1)))
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = self.cap.read()
        return fr if ok else None

    def close(self):
        self.cap.release()


def resolve_players(frame, conf=0.35):
    """-> {'left': (x1,y1,x2,y2) or None, 'right': ...}"""
    r = det.predict(frame, verbose=False, conf=conf)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return {"left": None, "right": None}

    xyxy = r.boxes.xyxy.cpu().numpy()
    cls  = r.boxes.cls.cpu().numpy().astype(int)

    tables = xyxy[cls == TABLE_CLS]
    mid = (tables[0][0] + tables[0][2]) / 2 if len(tables) else frame.shape[1] / 2

    players = xyxy[cls == PLAYER_CLS]
    if len(players) == 0:
        return {"left": None, "right": None}

    cx = (players[:, 0] + players[:, 2]) / 2
    left_side, right_side = players[cx < mid], players[cx >= mid]

    # furthest from centre on each side = the athlete, not the umpire
    left  = left_side[np.argmin((left_side[:, 0] + left_side[:, 2]) / 2)] if len(left_side) else None
    right = right_side[np.argmax((right_side[:, 0] + right_side[:, 2]) / 2)] if len(right_side) else None
    return {"left": left, "right": right}


def crop(frame, box, pad=CROP_PAD, out=(220, 300)):
    """Padded crop, letterboxed to a fixed size so tiles align."""
    if box is None:
        t = np.full((out[1], out[0], 3), 30, np.uint8)
        cv2.putText(t, "no det", (out[0]//2 - 42, out[1]//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 200), 2)
        return t

    H, W = frame.shape[:2]
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - w * pad));      x2 = int(min(W, x2 + w * pad))
    y1 = int(max(0, y1 - h * pad * 0.5)); y2 = int(min(H, y2 + h * pad * 0.3))
    c = frame[y1:y2, x1:x2]
    if c.size == 0:
        return np.full((out[1], out[0], 3), 30, np.uint8)

    scale = min(out[0] / c.shape[1], out[1] / c.shape[0])
    rs = cv2.resize(c, (int(c.shape[1] * scale), int(c.shape[0] * scale)))
    t = np.full((out[1], out[0], 3), 20, np.uint8)
    oy, ox = (out[1] - rs.shape[0]) // 2, (out[0] - rs.shape[1]) // 2
    t[oy:oy + rs.shape[0], ox:ox + rs.shape[1]] = rs
    return t


def label(img, text, y=22, colour=(0, 255, 255), scale=0.55):
    cv2.putText(img, text, (6, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4)
    cv2.putText(img, text, (6, y), cv2.FONT_HERSHEY_SIMPLEX, scale, colour, 1)
    return img

print("helpers ready")

helpers ready


## 5 · Build handedness sheets

Serves are sampled deliberately: the server holds the ball in the free hand and the bat in the playing hand, so the two are unambiguous. Falls back to any stroke if a side has no serves.

One JPG per video, ~2 min total.

In [7]:
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = None

hand_sheets = {}

for vid in VIDEO_IDS:
    vpath = VIDEOS / f"{vid}.mp4"
    if not vpath.exists():
        print(f"  {vid}: video missing, skipped"); continue

    sv = strokes[strokes.video_id == vid]
    rows = []
    for side in ["left", "right"]:
        pool = sv[(sv.side == side) & (sv.technique == "serve")]
        if len(pool) < N_HAND_FRAMES:                 # fall back to any stroke
            pool = pd.concat([pool, sv[sv.side == side]]).drop_duplicates("stroke_id")
        pick = pool.head(N_HAND_FRAMES * 4).sample(
            min(N_HAND_FRAMES, len(pool)), random_state=42)
        rows.append((side, pick.sort_values("frame_120")))

    rd = VideoReader(vpath)
    tiles = []
    for side, pick in rows:
        row = []
        for _, s in pick.iterrows():
            # +8 frames: just past contact, arm extended, bat clearly visible
            fr = rd.at(int(s.frame_120) + 8)
            if fr is None:
                row.append(crop(None, None)); continue
            boxes = resolve_players(fr)
            t = crop(fr, boxes[side])
            label(t, f"{side}  f{int(s.frame_120)}")
            label(t, s.technique, y=290, colour=(180, 255, 180), scale=0.5)
            row.append(t)
        while len(row) < N_HAND_FRAMES:
            row.append(crop(None, None))
        tiles.append(np.hstack(row))
    rd.close()

    sheet = np.vstack(tiles)
    banner = np.full((40, sheet.shape[1], 3), 25, np.uint8)
    label(banner, f"{vid}   (top row = LEFT player,  bottom row = RIGHT player)",
          y=27, scale=0.7)
    sheet = np.vstack([banner, sheet])

    out = SHEETS / f"handedness_{vid}.jpg"
    cv2.imwrite(str(out), sheet, [cv2.IMWRITE_JPEG_QUALITY, 92])
    hand_sheets[vid] = sheet
    print(f"  {vid}: written")

print(f"\n{len(hand_sheets)} sheets -> {SHEETS}")

  game_1: written
  game_2: written
  game_3: written
  game_4: written
  game_5: written
  test_1: written
  test_2: written
  test_3: written
  test_4: written
  test_5: written
  test_6: written
  test_7: written

12 sheets -> /content/drive/MyDrive/tt_coach/derived/meta/review_sheets


## 6 · Review

For each player ask: **which hand holds the bat?**

- Bat arm swings across the body and reaches furthest at contact
- The free hand is the one that tossed the ball on a serve
- If a crop is empty or shows the wrong person, note the video and fix its entry by eye in the next cell

Run the cell, scroll through all 12.

In [8]:
for vid in VIDEO_IDS:
    if vid not in hand_sheets:
        continue
    print("=" * 70); print(f"  {vid}"); print("=" * 70)
    s = hand_sheets[vid]
    w = 1100
    s = cv2.resize(s, (w, int(w * s.shape[0] / s.shape[1])))
    if cv2_imshow:
        cv2_imshow(s)
    else:
        from IPython.display import Image, display
        _, buf = cv2.imencode(".jpg", s); display(Image(data=buf.tobytes()))

Output hidden; open in https://colab.research.google.com to view.

## 7 · Record handedness

Edit the dict below — `"L"` or `"R"` for each player. Defaults are `"R"` since right-handers dominate; change only the ones you see are left-handed.

Per the source paper there are **three left-handed players, all in the test videos** — so expect ~3 edits, all in `test_*`. If you find none, re-check the test sheets before moving on.

Set `notes` for anything odd (unclear crop, wrong person detected, player changes between games).

In [9]:
HANDEDNESS = {
    "game_1": {"left": "R", "right": "R"},
    "game_2": {"left": "R", "right": "R"},
    "game_3": {"left": "R", "right": "R"},
    "game_4": {"left": "R", "right": "R"},
    "game_5": {"left": "R", "right": "R"},
    "test_1": {"left": "R", "right": "R"},
    "test_2": {"left": "R", "right": "R"},
    "test_3": {"left": "R", "right": "R"},
    "test_4": {"left": "R", "right": "R"},
    "test_5": {"left": "R", "right": "R"},
    "test_6": {"left": "R", "right": "R"},
    "test_7": {"left": "R", "right": "R"},
}

NOTES = {
    # "test_3": "left player crop unclear on frame 2 - confirmed L from sheet 3",
}

## 8 · Validate & write `players.json`

In [10]:
assert set(HANDEDNESS) == set(VIDEO_IDS), \
    f"video mismatch: {set(VIDEO_IDS) ^ set(HANDEDNESS)}"

players, n_left = {}, 0
for vid in VIDEO_IDS:
    for side in ["left", "right"]:
        h = str(HANDEDNESS[vid][side]).strip().upper()
        assert h in ("L", "R"), f"{vid}/{side}: expected 'L' or 'R', got {h!r}"
        n_left += h == "L"
        sv = strokes[(strokes.video_id == vid) & (strokes.side == side)]
        players[f"{vid}__{side}"] = {
            "video_id": vid, "side": side, "handedness": h,
            "fold": folds["video2fold"][vid],
            "n_strokes": int(len(sv)),
            "class_counts": sv.shot_class.value_counts().to_dict(),
            "note": NOTES.get(vid, ""),
        }

(META / "players.json").write_text(json.dumps(players, indent=2))

print(f"{len(players)} player-instances, {n_left} left-handed\n")
lefties = {k: v for k, v in players.items() if v["handedness"] == "L"}
if lefties:
    print("left-handed:")
    for k, v in lefties.items():
        print(f"  {k:<22} fold {v['fold']}  {v['n_strokes']} strokes")
    tr = [k for k, v in lefties.items() if v["video_id"].startswith("game")]
    print(f"\n  in TRAINING videos: {len(tr)}  {tr if tr else '(none - as the paper reports)'}")
else:
    print("  !! No left-handers found. The paper reports three, all in test")
    print("     videos. Re-check the test_* sheets before continuing.")

print(f"\n-> {META / 'players.json'}")

24 player-instances, 0 left-handed

  !! No left-handers found. The paper reports three, all in test
     videos. Re-check the test_* sheets before continuing.

-> /content/drive/MyDrive/tt_coach/derived/meta/players.json


---
## Part B — Golden set

40 strokes, stratified across class and fold. For each you see 5 frames spanning the window, with the contact frame marked.

Three things to check per stroke:
1. **Class** — does the labelled class match what you see?
2. **Contact frame** — is the marked frame actually racket-ball impact?
3. **Attribution** — is the labelled side the player who actually hit it?

~10 min to build.

In [11]:
# Stratified so every (fold x class) cell that exists gets at least one
# example, then filled proportionally to reach N_GOLDEN. A pure per-cell
# quota over-weights tiny cells and can miss whole folds.
cells_ = strokes.groupby(["fold", "shot_class"]).size()
cells_ = cells_[cells_ > 0]

picks = []
for (f, c), n in cells_.items():
    g = strokes[(strokes.fold == f) & (strokes.shot_class == c)]
    picks.append(g.sample(1, random_state=42))
golden = pd.concat(picks)

remaining = N_GOLDEN - len(golden)
if remaining > 0:
    rest = strokes[~strokes.stroke_id.isin(golden.stroke_id)]
    # proportional to class frequency, so the common classes get more eyes
    extra = (rest.groupby("shot_class", group_keys=False)
                 .apply(lambda g: g.sample(
                     max(1, int(round(remaining * len(g) / len(rest)))),
                     random_state=42)))
    golden = pd.concat([golden, extra.head(remaining)])

golden = golden.sort_values(["video_id", "frame_120"]).reset_index(drop=True)

print(f"{len(golden)} strokes sampled")
print(pd.crosstab(golden.fold, golden.shot_class).to_string())
print(f"\nvideos covered: {golden.video_id.nunique()}/{strokes.video_id.nunique()}")
print(f"cells covered : {len(golden.groupby(['fold','shot_class']))}/{len(cells_)}")

39 strokes sampled
shot_class  attack  control  defence  serve
fold                                       
A                1        1        1      2
B                3        2        1      1
C                2        1        1      1
D                1        2        3      1
E                3        1        1      1
F                1        1        1      2
G                1        1        1      1

videos covered: 10/12
cells covered : 28/28


/tmp/ipykernel_1099/1587156405.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


### Build strips

In [12]:
OFF = np.linspace(-folds["window"]["pre"], folds["window"]["post"],
                  GOLDEN_STRIP).astype(int)
golden_strips = {}

for vid, grp in golden.groupby("video_id"):
    vpath = VIDEOS / f"{vid}.mp4"
    if not vpath.exists():
        continue
    rd = VideoReader(vpath)
    for _, s in grp.sort_values("frame_120").iterrows():
        row = []
        for o in OFF:
            fr = rd.at(int(s.frame_120) + int(o))
            if fr is None:
                row.append(crop(None, None)); continue
            t = crop(fr, resolve_players(fr)[s.side])
            is_contact = o == 0
            label(t, f"{o:+d}", colour=(0, 0, 255) if is_contact else (200, 200, 200))
            if is_contact:
                cv2.rectangle(t, (0, 0), (t.shape[1] - 1, t.shape[0] - 1), (0, 0, 255), 3)
            row.append(t)
        strip = np.hstack(row)
        def _s(v):                      # NaN-safe for optional label fields
            return "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)
        banner = np.full((34, strip.shape[1], 3), 25, np.uint8)
        label(banner, f"{s.stroke_id}   {_s(s.side)} {_s(s.high_level)} "
                      f"{_s(s.technique)}   ->  {_s(s.shot_class).upper()}",
              y=24, scale=0.6)
        golden_strips[s.stroke_id] = np.vstack([banner, strip])
    rd.close()
    print(f"  {vid}: {len(grp)} strips")

print(f"\n{len(golden_strips)} strips built")

  game_1: 5 strips
  game_2: 7 strips
  game_3: 5 strips
  game_4: 7 strips
  game_5: 6 strips
  test_1: 1 strips
  test_2: 1 strips
  test_3: 2 strips
  test_4: 4 strips
  test_7: 1 strips

39 strips built


### Review

Note any `stroke_id` where the class looks wrong, the red-boxed frame isn't contact, or the wrong player is shown.

In [13]:
for sid, strip in golden_strips.items():
    w = 1100
    s = cv2.resize(strip, (w, int(w * strip.shape[0] / strip.shape[1])))
    if cv2_imshow:
        cv2_imshow(s)
    else:
        from IPython.display import Image, display
        _, buf = cv2.imencode(".jpg", s); display(Image(data=buf.tobytes()))

Output hidden; open in https://colab.research.google.com to view.

### Record disagreements

List only the ones you disagree with — everything unlisted counts as verified.

In [14]:
# stroke_id -> what's wrong
DISAGREE = {
    # "game_1_0002245": "class looks like control, not attack",
    # "test_4_0012880": "contact is ~6 frames late",
    # "game_3_0031002": "wrong player shown",
}

verdict = {
    sid: {"ok": sid not in DISAGREE, "issue": DISAGREE.get(sid, "")}
    for sid in golden_strips
}
n_ok = sum(v["ok"] for v in verdict.values())

(META / "golden_set.json").write_text(json.dumps({
    "n": len(verdict),
    "n_ok": n_ok,
    "agreement": round(n_ok / max(len(verdict), 1), 3),
    "strokes": verdict,
}, indent=2))

print(f"{n_ok}/{len(verdict)} verified  ({100*n_ok/max(len(verdict),1):.0f}% agreement)")
if DISAGREE:
    print("\nflagged:")
    for k, v in DISAGREE.items():
        print(f"  {k}: {v}")
print(f"\n-> {META / 'golden_set.json'}")

39/39 verified  (100% agreement)

-> /content/drive/MyDrive/tt_coach/derived/meta/golden_set.json


---
## Done

| artifact | used by |
|---|---|
| `derived/meta/players.json` | Stage 3 — handedness mirroring |
| `derived/meta/golden_set.json` | Stage 4 — debugging oracle |
| `derived/meta/review_sheets/` | reference |

**Reading the agreement rate:** below ~90% means label noise is capping what any model can achieve, and the confusions you flagged are worth checking against the fold-level confusion matrix later. Push/block boundary cases are the expected source.

To change a handedness call later, edit `players.json` directly — no rerun needed.

Next: **`02_pose_extraction.ipynb`** (Stage 2), ~1.25 h on the T4.